# Difference in Normalized Burn Ratio (dNBR) for Fire Area Detection using OpenEO

This notebook demonstrates how to detect the burned area in a forest fire using Sentinel-2 imagery. The burned area is identifyed based on the difference in normalized burn ratio. A pre and a post fire image are loaded and the dNBR is calculated for each of them and then subtracted.

FR's comments: 
- Can you add the main ideas of your internship project here? Archaic goal: Impact of Portugal forest fire. Key processes: 1) Calculate dNBR between two periods of time where fire happens, 2) Identify the tree composition of the affected forest area using ML, 3) Register a UDF for ML, 4) Create a storyline powered by EOPF openEO. 
- Would be nice if you could add a graph like the following example: https://github.com/Open-EO/openeo-community-examples/blob/main/python/DynamicLandCoverMapping/Dynamic%20land%20cover%20mapping.ipynb (not urgent, but good to keep in mind for your final presentation)
- See my comments below, signify by ##. Summaries:
- Remove unused libraries? in `1. Import required libraries` section
- I moved your parameter definition to `dNBR_Portugal.params.py` so that we keep the focus of the notebook on processing pipeline. Check if I input your parameter definition correctly in `dNBR_Portugal.params.py` and delete the commented out cell. Section: `### Define Parameters and Connect to OpenEO Backend`
- In `## 3. Calculate dNBR by subtracting between the cubes` section, I created a function called `calculate_dnbr_and_mask` which combines dNBR calculation and its masking. I did it to simplify the process. The function returns a tuple of two arrays which you can download separately. 
- In `## 4. Visualize dNBR` section, I put the filtering method, with a philosophy that this is a processing step before plotting. Then, I plot the original dNBR, masked dNBR, and the clean-up version of masked dNBR in a faceted grid so that people can see the differences at once.
- I asked a question at the end of section 4
- What you can do now: address my comments, run the notebook, and delete my comments when everything runs correctly.

# Overview

1. Import libraries, define parameters and connect to OpenEO backend
2. Load pre and post fire satellite image
3. Calculate and visualize dNBR 
4. Failed Approaches to load and index both images in one cube

## 1. Import required libraries

We begin by importing the necessary Python libraries for data processing and visualization.

In [ ]:
## Can you remove unused libraries?

import matplotlib.pyplot as plt
from PIL import Image
import openeo
from openeo.processes import array_create, if_, and_ # Remove if unused
from openeo.api.process import Parameter
# OpenEO UDP parameter management system 
from openeo_udp import ParameterManager # Remove if unused
import rasterio # Remove if unused
from scipy import ndimage
import numpy as np


### Define Parameters and Connect to OpenEO Backend

Load algorithm parameters from the co-located parameter file and connect to an OpenEO backend with automatic endpoint selection. Choose between CDE and EOPF

In [ ]:
## I commented out the following cell, you can review and delete it after
## This is because I moved your parameter definition to dNBR_Portugal.params.py
## Then I import them programmatically in the following cells

In [ ]:
# connection = openeo.connect(
#     url="https://openeo.dataspace.copernicus.eu/"
# ).authenticate_oidc()

# fire_date = 2025 # 2024 or 2025
# fire_bands = ["B8A", "B12"]

# if fire_date == 2025:
#     time_post= ["2025-10-15", "2025-10-16"]
#     time_pre = ["2025-07-25", "2025-07-31"]

#     current_params = {
#             "location_name": "Forest, North Portugal",
#             "bounding_box": Parameter(
#                 "bounding_box",
#                 description="Fire area in 2025",
#                 default= {"west": -7.987061, "south": 40.012891, "east": -7.434998,"north": 40.359103}
#     ),
#             "time": Parameter(
#                 "time",
#                 description="Temporal range for data acquisition",
#                 # default=["2025-10-15", "2025-10-16"], # + last
#                 default=["2025-07-25", "2025-07-31"],  # + last
#             ),
#             "bands": Parameter(
#                 "bands",
#                 description="Sentinel-2 bands required for APA calculation",
#                 default=fire_bands
#     #,
#             ),
#             "collection": Parameter(
#                 "collection",
#                 description="Data collection identifier",
#                 default="SENTINEL2_L2A",
#             ),
#             "cloud_cover": Parameter(
#                 "cloud_cover",
#                 description="Maximum cloud cover percentage",
#                 default=30,
#             ),
#         }


# if fire_date == 2024:
    
#     time_post= ["2024-09-30", "2024-10-15"] # + last
#     time_pre=  ["2024-08-20", "2024-09-01"]  # + last

#     current_params = {
#             "location_name": "Reriz e Gafanhao, North Portugal",
#             "bounding_box": Parameter(
#                 "bounding_box",
#                 description="Fire area in 2024",
#                 default= {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}
#     ),
#             "time": Parameter(
#                 "time",
#                 description="Temporal range for data acquisition",
#                 # default=["2024-09-30", "2024-10-30"], # + last
#                 default=["2024-08-01", "2024-09-01"],  # + last
#             ),
#             "bands": Parameter(
#                 "bands",
#                 description="Sentinel-2 bands required for APA calculation",
#                 default=["B8A", "B12"]
#     #,
#             ),
#             "collection": Parameter(
#                 "collection",
#                 description="Data collection identifier",
#                 default="SENTINEL2_L2A",
#             ),
#             "cloud_cover": Parameter(
#                 "cloud_cover",
#                 description="Maximum cloud cover percentage",
#                 default=30,
#             ),
#         }

In [ ]:
# Initialize parameter manager
param_manager = ParameterManager('dNBR_Portugal.params.py')

# Display available options using the built-in helper
param_manager.print_options("dNBR algorithm")

In [ ]:
# Connect using the a parameter set for a specified location on the Copernicus Data Space endpoint
connection, current_params = param_manager.quick_connect(
    param_set="forest_north_portugal_2025",
    endpoint="copernicus_dataspace", # Connecting to CDSE
)

## 2. Load image before and after the fire as seperate cubes

In [ ]:
# First define functions to use for loading image and calculating NBR
def load_and_sample(params, time, res=100):
    """
    Input: Set of parameters, time frame, desired resolution
    Output: Cube that contains the last image in the time frame 
    """ 
    s2cube = connection.load_collection(
    params["collection"].default,
    temporal_extent=time,
    spatial_extent=params["bounding_box"].default,
    bands=params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= params["cloud_cover"].default,
    },
    )

    s2cube = s2cube.reduce_dimension(dimension="t", reducer="last")

    s2cube = s2cube.resample_spatial(
        resolution=[res, res],
      method="near",
    )

    return s2cube

In [ ]:
# Define Resolution
res_dnbr = 20

# Load Satellite image before and after the fire as cubes
s2cube_pre = load_and_sample(current_params, current_params["time_pre"].default, res_dnbr)
s2cube_post = load_and_sample(current_params, current_params["time_post"].default, res_dnbr)

## 3. Calculate dNBR by subtracting between the cubes

In [ ]:
### MAIN PROCESS ###
def calculate_nbr(data):
     """ 
     Input: Cube with bands 08 and 12
     Output: NBR
     """
     B08, B12 = (
          data[0],
          data[1],
     )
     nbr = (B08 - B12) / (B08 + B12)
     return nbr

## I combined dNBR calculation and masking to values above 0.3 to simplify process
def calculate_dnbr_and_mask(nbr_pre, nbr_post, burn_threshold):
    dnbr = nbr_pre - nbr_post
    dnbr_mask = dnbr > burn_threshold
    return dnbr, dnbr_mask

In [ ]:
# Calculate NBR of pre and post cube
nbr_post = s2cube_post.reduce_dimension(dimension="bands", reducer=calculate_nbr)
nbr_pre = s2cube_pre.reduce_dimension(dimension="bands", reducer=calculate_nbr)

burn_treshold_default = 0.3
dnbr, dnbr_mask = calculate_dnbr_and_mask(nbr_pre, nbr_post, burn_treshold_default)

In [ ]:
# Download dNBR
dnbr_vis = dnbr.linear_scale_range(-1, 1, 0, 255)
dnbr_vis = dnbr_vis.save_result("PNG")

filename = f"apa_{current_params['location_name'].replace(' ', '_').replace(',', '').lower()}.png"

connection.download(
    {
        "process_graph": dnbr_vis.flat_graph(),
    }, filename,
)

# Download masked dNBR
mask_result = dnbr_mask.save_result("GTiff")

filename_mask = f"firemask_{current_params['location_name'].replace(' ', '_').replace(',', '').lower()}.tif"

connection.download(
    {
        "process_graph": mask_result.flat_graph(),
    }, filename_mask
)

## 4. Visualize dNBR

In [ ]:
# Importing the downloaded data
dnbr_img = Image.open(filename)
mask_arr = np.array(Image.open(filename_mask))

# Apply connected component labeling to the mask to keep only the largest connected component -> for identifying main area of fire
labeled, n = ndimage.label(mask_arr, structure=np.ones((3, 3)))
sizes = ndimage.sum(mask_arr, labeled, range(1, n + 1))
clean_mask = (labeled == sizes.argmax() + 1).astype("uint8")

In [ ]:
# Plotting original dNBR, masked dNBR, and a clean-up version of masked dNBR side by side
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax1, ax2, ax3, ax4 = axes.flat
fig.delaxes(ax4)

im1 = ax1.imshow(dnbr_img, cmap="seismic", vmin=0, vmax=255)
fig.colorbar(im1, ax=ax1, label="dNBR", shrink=0.8)
ax1.set_title(f"dNBR\n{current_params['location_name']}")
ax1.axis("off")

ax2.imshow(mask_arr, cmap="gray")
ax2.set_title(f"Burned Area (dNBR > {burn_treshold_default})\n{current_params['location_name']}")
ax2.axis("off")

ax3.imshow(clean_mask, cmap='gray')
ax3.set_title(f"dNBR > {burn_treshold_default} mask\ncleaned up to focus on the largest connected component")
ax3.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## NOTES
# # Using CDE either of these three work
# # FR: Can the arithmetic subtraction work in EOPF? 

# # Arithmetic
# dnbr = nbr_pre - nbr_post

# # With subtract process
# # from openeo.processes import subtract
# # dnbr = nbr_pre.subtract(nbr_post)

# # With merge_cubes
# # dnbr = nbr_pre.merge_cubes(nbr_post, overlap_resolver="subtract")

----

## 4. Approach 2: Load Both images in one cube

FR: Do we still need this?

In [ ]:
s2cube_both = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=["2025-07-25", "2025-10-16"],  # full range
    spatial_extent=current_params["bounding_box"].default,
    bands=current_params["bands"].default,
)
s2cube_both = s2cube_both.resample_spatial(resolution=[100, 100], method="near")

# Apply NBR to get time series
nbr_timeseries = s2cube_both.apply_dimension(dimension="bands", process=calculate_nbr)

#### Custom Reducer - Return dnbr
I would like to index nbr_timeseries [0] and [-1] (first and last element to perform calculation)
However, this seems to fail because I can not index -1, is there another way for me to find the last element?

In [ ]:
def dnbr_reducer(data):
    # data is the time series of NBR values
    first_nbr = data[0]
    last_nbr = data[-1]
    dnbr =  first_nbr - last_nbr
    return dnbr

# Reduce time dimension with your custom reducer (returns single image)
dnbr = nbr_timeseries.reduce_dimension(dimension="t", reducer=dnbr_reducer)

In [ ]:
test(dnbr)

Testing if this works for first and second image

In [ ]:
def dnbr_reducer(data):
    # data is the time series of NBR values
    first_nbr = data[0]
    second_nbr = data[1]
    dnbr =  first_nbr - second_nbr
    return dnbr

# Reduce time dimension with your custom reducer (returns single image)
dnbr = nbr_timeseries.reduce_dimension(dimension="time", reducer=dnbr_reducer)


In [ ]:
test(dnbr)